# BoostFL-IoT ablation study

## 1. Imports

In [38]:
import random
from dataclasses import dataclass
from typing import List, Dict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, recall_score, f1_score

import warnings

warnings.filterwarnings("ignore")

SEED = 42


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


## 2. Configuration

In [39]:
NUM_CLIENTS = 10
NUM_PARTITIONS = 10
NUM_ROUNDS = 8
BATCH_SIZE = 32
EPOCHS = 5
LEARNING_RATE = 1e-3
SERVER_LR = 1.0

EPSILON = 1e-8
MIN_ALPHA = 0.1
MAX_ALPHA = 10.0

DIRICHLET_ALPHA = 0.08

## 3. Data loading (BoT-IoT)

In [40]:
def load_botiot_dataset(file_path, target_column="category",
                        test_size=0.2, random_state=SEED):
    df = pd.read_csv(file_path).drop_duplicates()
    df = df.dropna(subset=[target_column])

    num_cols = df.select_dtypes(include=[np.number]).columns
    cat_cols = df.select_dtypes(exclude=[np.number]).columns
    for col in num_cols:
        if df[col].isnull().any():
            df[col] = df[col].fillna(df[col].median())
    for col in cat_cols:
        if df[col].isnull().any():
            mode = df[col].mode()
            df[col] = df[col].fillna(mode[0] if not mode.empty else "Unknown")

    y = df[target_column].astype(str).str.strip()
    cols_to_drop = [target_column, "subcategory ", "attack", "category",
                    "pkSeqID", "saddr", "daddr", "soui", "doui",
                    "sco", "dco", "smac", "dmac"]
    X = df.drop(columns=cols_to_drop, errors="ignore").copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y)

    non_numeric = list(
        set(X_train.select_dtypes(exclude=[np.number]).columns)
        | set(X_test.select_dtypes(exclude=[np.number]).columns))
    for col in non_numeric:
        le = LabelEncoder()
        le.fit(X_train[col].astype(str))
        mapping = {c: i for i, c in enumerate(le.classes_)}
        X_train[col] = X_train[col].astype(str).map(mapping).fillna(-1).astype(int)
        X_test[col] = X_test[col].astype(str).map(mapping).fillna(-1).astype(int)

    def safe_numeric(d):
        d = d.apply(lambda col: col.map(lambda v: str(v).strip() if isinstance(v, str) else v))
        d = d.apply(pd.to_numeric, errors="coerce")
        return d.replace([np.inf, -np.inf], np.nan).fillna(0)

    X_train = safe_numeric(X_train)
    X_test = safe_numeric(X_test)

    label_encoder = LabelEncoder()
    y_train_enc = label_encoder.fit_transform(y_train.values)
    y_test_enc = label_encoder.transform(y_test.values)
    class_names = label_encoder.classes_
    num_classes = len(class_names)
    input_dim = X_train.shape[1]

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train.values.astype(np.float64))
    X_test_s = scaler.transform(X_test.values.astype(np.float64))

    train_ds = TensorDataset(torch.from_numpy(X_train_s).float(),
                             torch.from_numpy(y_train_enc).long())
    test_ds = TensorDataset(torch.from_numpy(X_test_s).float(),
                            torch.from_numpy(y_test_enc).long())
    return train_ds, test_ds, class_names, num_classes, input_dim


DATA_CSV = "../../data/Bot-IoT.csv"
train_dataset, test_dataset, CLASS_NAMES, NUM_CLASSES, INPUT_DIM = load_botiot_dataset(DATA_CSV)
print(f"Features: {INPUT_DIM} | Train: {len(train_dataset)} | Test: {len(test_dataset)}")

Features: 23 | Train: 40904 | Test: 10226


## 4. Non-IID partitioning

In [41]:
def partition_dirichlet(dataset, num_partitions, num_classes, alpha, seed=SEED):
    rng = np.random.RandomState(seed)
    labels = np.array([dataset[i][1] for i in range(len(dataset))])
    idx_by_class = [np.where(labels == c)[0].tolist() for c in range(num_classes)]
    partitions = [[] for _ in range(num_partitions)]
    for c in range(num_classes):
        idx = idx_by_class[c]
        rng.shuffle(idx)
        props = rng.dirichlet([alpha] * num_partitions)
        counts = (props * len(idx)).astype(int)
        diff = len(idx) - counts.sum()
        for k in (np.argsort(props)[-diff:] if diff > 0 else []):
            counts[k] += 1
        start = 0
        for p in range(num_partitions):
            end = start + counts[p]
            partitions[p].extend(idx[start:end])
            start = end
    for p in range(num_partitions):
        rng.shuffle(partitions[p])
    return partitions


TRAIN_PARTITIONS = partition_dirichlet(
    train_dataset, NUM_PARTITIONS, NUM_CLASSES, DIRICHLET_ALPHA)

## 5. Weak learner and residual loss

In [42]:
class WeakLearner(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 50)
        self.fc2 = nn.Linear(50, 25)
        self.fc3 = nn.Linear(25, num_classes)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


class ResidualLoss(nn.Module):
    def forward(self, predictions, residuals):
        return F.smooth_l1_loss(predictions, residuals)

## 6. Ablation configuration

In [43]:
@dataclass
class AblationConfig:
    name: str
    use_residual_learning: bool = True
    aggregation: str = "boosting"
    use_ensemble: bool = True

## 7. Ensemble

In [44]:
class Ensemble:
    def __init__(self, f0, device, use_ensemble=True):
        self.f0 = f0.to(device)
        self.device = device
        self.use_ensemble = use_ensemble
        self.learners = []
        self.alphas = []

    def add_learner(self, params, alpha):
        m = WeakLearner(INPUT_DIM, NUM_CLASSES).to(self.device)
        sd = m.state_dict()
        new_sd = {k: (torch.tensor(v) if isinstance(v, np.ndarray) else v).to(self.device)
                  for k, v in zip(sd.keys(), params)}
        m.load_state_dict(new_sd)
        m.eval()
        if self.use_ensemble:
            self.learners.append(m)
            self.alphas.append(alpha)
        else:
            self.learners = [m]
            self.alphas = [alpha]

    @torch.no_grad()
    def predict(self, x):
        out = self.f0.unsqueeze(0).expand(x.size(0), -1).clone()
        if self.learners and self.alphas:
            total = sum(self.alphas)
            if total > EPSILON:
                acc = torch.zeros_like(out)
                for m, a in zip(self.learners, self.alphas):
                    acc += (a / total) * m(x)
                out = out + acc
        return out

## 8. Client local update

In [45]:
def local_update(cfg, global_params, ensemble, client_idx):
    partition = TRAIN_PARTITIONS[client_idx]
    loader = DataLoader(Subset(train_dataset, partition),
                        batch_size=BATCH_SIZE, shuffle=True)

    net = WeakLearner(INPUT_DIM, NUM_CLASSES).to(DEVICE)
    sd = net.state_dict()
    net.load_state_dict({k: torch.tensor(v).to(DEVICE)
                         for k, v in zip(sd.keys(), global_params)})
    net.train()

    optimizer = optim.Adam(net.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    residual_loss_fn = ResidualLoss()
    ce_loss_fn = nn.CrossEntropyLoss()

    total_loss, n_batches = 0.0, 0
    for _ in range(EPOCHS):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            logits = net(xb)
            if cfg.use_residual_learning:
                with torch.no_grad():
                    ens_probs = torch.softmax(ensemble.predict(xb), dim=1)
                    y_oh = torch.zeros(yb.size(0), NUM_CLASSES, device=DEVICE)
                    y_oh.scatter_(1, yb.unsqueeze(1), 1)
                    residuals = y_oh - ens_probs
                loss = residual_loss_fn(logits, residuals)
            else:
                loss = ce_loss_fn(logits, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_batches += 1

    avg_loss = min(max(total_loss / max(n_batches, 1), EPSILON), 10.0)
    alpha = 1.0 / (1.0 + avg_loss)
    alpha = max(MIN_ALPHA, min(MAX_ALPHA, alpha))

    new_params = [v.detach().cpu().numpy() for v in net.state_dict().values()]
    return new_params, len(partition), alpha

## 9. Server aggregation

In [46]:
def aggregate(cfg, global_params, client_results):
    updates = [np.zeros_like(p) for p in global_params]
    if cfg.aggregation == "boosting":
        weights = [a for (_, _, a) in client_results]
    elif cfg.aggregation == "fedavg":
        weights = [float(n) for (_, n, _) in client_results]
    else:
        raise ValueError(f"Unknown aggregation '{cfg.aggregation}'")

    wsum = sum(weights)
    for (params, _, _), w in zip(client_results, weights):
        for i in range(len(updates)):
            updates[i] += w * (params[i] - global_params[i])
    if wsum > EPSILON:
        updates = [(u / wsum) * SERVER_LR for u in updates]
    else:
        updates = [np.zeros_like(p) for p in global_params]

    new_global = [gp + updates[i] for i, gp in enumerate(global_params)]
    mean_alpha = float(np.mean([a for (_, _, a) in client_results]))
    return new_global, mean_alpha

## 10. Evaluation and driver

In [47]:
@torch.no_grad()
def evaluate_ensemble(ensemble):
    loader = DataLoader(test_dataset, batch_size=256, shuffle=False)
    y_true, y_pred = [], []
    for xb, yb in loader:
        xb = xb.to(DEVICE)
        logits = ensemble.predict(xb)
        y_pred.extend(torch.argmax(logits, 1).cpu().numpy())
        y_true.extend(yb.numpy())
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }


def run_federated(cfg, num_rounds=NUM_ROUNDS):
    set_seed(SEED)
    init_model = WeakLearner(INPUT_DIM, NUM_CLASSES).to(DEVICE)
    global_params = [v.cpu().numpy() for v in init_model.state_dict().values()]
    f0 = torch.randn(NUM_CLASSES, device=DEVICE) * 0.01
    ensemble = Ensemble(f0, DEVICE, use_ensemble=cfg.use_ensemble)

    for _ in range(num_rounds):
        client_results = []
        for cid in range(NUM_CLIENTS):
            if len(TRAIN_PARTITIONS[cid]) == 0:
                continue
            client_results.append(local_update(cfg, global_params, ensemble, cid))
        global_params, mean_alpha = aggregate(cfg, global_params, client_results)
        ensemble.add_learner(global_params, mean_alpha)

    return evaluate_ensemble(ensemble)

## 11. Ablation configurations

In [48]:
ablations = [
    AblationConfig("BoostFL-IoT"),
    AblationConfig("No res.", use_residual_learning=False),
    AblationConfig("FedAvg agg.", aggregation="fedavg", use_ensemble=True),
]

## 12. Run

In [49]:
results = {}
for cfg in ablations:
    results[cfg.name] = run_federated(cfg)
    print(f"{cfg.name}: acc={results[cfg.name]['accuracy']:.4f} "
          f"f1={results[cfg.name]['f1']:.4f}")

BoostFL-IoT: acc=0.9723 f1=0.9522
No res.: acc=0.9449 f1=0.9283
FedAvg agg.: acc=0.5802 f1=0.4095


## 13. Results

In [50]:
ref_acc = results["BoostFL-IoT"]["accuracy"]
rows = []
for cfg in ablations:
    m = results[cfg.name]
    rows.append({
        "Configuration": cfg.name,
        "Accuracy": round(m["accuracy"], 4),
        "Recall": round(m["recall"], 4),
        "F1-score": round(m["f1"], 4),
        "Delta Acc.": round(m["accuracy"] - ref_acc, 4),
    })
table = pd.DataFrame(rows)
table

,Configuration,Accuracy,Recall,F1-score,Delta Acc.
0,BoostFL-IoT,0.9723,0.9340,0.9522,0.0000
1,No res.,0.9449,0.9109,0.9283,-0.0274
2,FedAvg agg.,0.5802,0.4788,0.4095,-0.3921
